# 06 · Bonus — heatmaps, correlation & a PCA ordination

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: R* - Instructor: Anderson Santos

## Setup — run this first

This cell installs the R packages the lessons use and makes the example data available. On **Google Colab** it just works: run it (Shift+Enter) and wait until it prints **Ready**. You do not need to understand it.

In [ ]:
# Setup — run this first. Works on Google Colab and on a local Jupyter with R.
# It installs the packages the lessons use and makes the example data available.

pkgs <- c("readr", "dplyr", "tidyr", "tibble", "ggplot2", "forcats", "stringr", "vegan")
to_install <- setdiff(pkgs, rownames(installed.packages()))
if (length(to_install) > 0) {
  # On Colab (Linux) use Posit Public Package Manager binaries: ~1-2 min instead of
  # compiling from source (~15-20 min).
  if (grepl("linux", R.version$os) && file.exists("/etc/os-release")) {
    cn <- grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE)
    cn <- gsub('VERSION_CODENAME=|"', "", cn)
    if (length(cn) == 1 && nzchar(cn)) {
      options(repos = c(CRAN = sprintf("https://p3m.dev/cran/__linux__/%s/latest", cn)))
      options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
              paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
    }
  }
  install.packages(to_install)
}

# Fetch the example data if it is not already next to the notebook (the Colab case).
if (!file.exists("../data/sample_metadata.csv") && !file.exists("data/sample_metadata.csv")) {
  system("git clone -q https://github.com/andersonavilasantos/ufz-summerschool-r.git course_repo")
  setwd("course_repo/notebooks")
}

# Load the packages quietly and keep read_csv output tidy (cleaner notebook output).
suppressPackageStartupMessages(invisible(lapply(pkgs, require, character.only = TRUE)))
options(readr.show_col_types = FALSE)

cat("Ready. Data folder:", if (file.exists("../data/sample_metadata.csv")) "../data" else "data", "\n")

## Learning objectives (time-buffer lesson)

Use this if the core session finishes early. You will:

- Draw an **abundance heatmap** of top genera with `geom_tile()`.
- Compute and visualise a **correlation** structure.
- Run a **PCA** ordination with base R (`prcomp`) — no Bioconductor.

All CRAN-only. Nothing here is new machinery — just new combinations.

---

## 0 · Setup

In [ ]:
library(readr); library(dplyr); library(tidyr)   # import, wrangle, reshape
library(ggplot2); library(forcats); library(tibble)  # plot, reorder factors, rownames helpers

# Works whether the notebook runs from notebooks/ or from the project root.
data_dir <- if (file.exists("../data/sample_metadata.csv")) "../data" else "data"
meta     <- read_csv(file.path(data_dir, "sample_metadata.csv"))  # one row per sample (participant)
taxonomy <- read_csv(file.path(data_dir, "taxonomy.csv"))         # genus -> family / phylum lookup
abund    <- read_csv(file.path(data_dir, "genus_abundance.csv"))  # wide: one column per genus

# Reshape wide -> long (tidyverse verbs want long data), then attach each genus's phylum.
ab_long <- abund |>
  pivot_longer(-sample_id, names_to = "genus", values_to = "abundance") |>  # every column except sample_id becomes rows
  left_join(taxonomy, by = "genus")   # + phylum / family columns (left_join keeps all abundance rows)

---

## 1 · A heatmap of the most abundant genera

**Idea:** pick the 20 most abundant genera overall, average their relative
abundance per region, and draw a tile grid — colour = abundance.

In [ ]:
# Step 1: find the 20 genera with the highest total abundance across the whole study.
top_genera <- ab_long |>
  group_by(genus) |>                     # one group per genus
  summarise(total = sum(abundance)) |>   # collapse each genus to its grand total
  slice_max(total, n = 20) |>            # keep the 20 largest (dplyr's top-n helper)
  pull(genus)                            # pull() extracts that column as a plain vector

# Step 2: for those genera, compute each one's mean relative abundance per region.
heat <- ab_long |>
  filter(genus %in% top_genera) |>                                  # keep only the top 20
  left_join(select(meta, sample_id, nationality), by = "sample_id") |>  # attach each sample's region
  group_by(sample_id) |>
  mutate(rel = abundance / sum(abundance)) |>   # within-sample proportion (removes depth differences)
  group_by(nationality, genus) |>
  summarise(mean_rel = mean(rel), .groups = "drop")   # average the proportion within each region

# Step 3: draw the tile grid. x = region, y = genus, fill = mean proportion.
ggplot(heat, aes(x = nationality,
                 y = fct_reorder(genus, mean_rel, sum),   # order genera by overall abundance (tidy y-axis)
                 fill = mean_rel)) +
  geom_tile(colour = "white") +                                   # one coloured tile per (region, genus)
  scale_fill_viridis_c(option = "C", labels = scales::percent) +  # perceptually-uniform colour, % labels
  labs(title = "Top 20 genera across regions",
       x = NULL, y = NULL, fill = "Mean\nrel. abund.") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))         # tilt region labels so they don't collide

**A second angle — the same tiles, but per BMI group.** A heatmap is just a
summary table with colour: swap the grouping variable and you ask a different
biological question with almost the same code.

In [ ]:
# Same top genera, but now average their relative abundance across BMI groups.
heat_bmi <- ab_long |>
  filter(genus %in% top_genera) |>
  left_join(select(meta, sample_id, bmi_group), by = "sample_id") |>  # attach BMI instead of region
  filter(!is.na(bmi_group)) |>                                        # drop samples with no BMI label
  group_by(sample_id) |>
  mutate(rel = abundance / sum(abundance)) |>                         # within-sample proportion again
  group_by(bmi_group, genus) |>
  summarise(mean_rel = mean(rel), .groups = "drop")

ggplot(heat_bmi, aes(x = bmi_group,
                     y = fct_reorder(genus, mean_rel, sum),
                     fill = mean_rel)) +
  geom_tile(colour = "white") +
  scale_fill_viridis_c(option = "C", labels = scales::percent) +
  labs(title = "Top 20 genera across BMI groups",
       x = NULL, y = NULL, fill = "Mean\nrel. abund.") +
  theme_minimal(base_size = 11) +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

> **Instructor note.** `viridis` palettes are perceptually uniform and
> colour-blind-friendly, a good default for heatmaps. `scale_fill_viridis_c()`
> ships with ggplot2, no extra package needed.

---

## 2 · Correlation between the major phyla

Do phyla trade off against each other across samples? Build a sample × phylum
relative-abundance matrix and correlate the columns.

In [ ]:
# Build a sample x phylum matrix of relative abundances.
phy_wide <- ab_long |>
  group_by(sample_id, phylum) |>
  summarise(a = sum(abundance), .groups = "drop") |>  # total reads per phylum in each sample
  group_by(sample_id) |>
  mutate(rel = a / sum(a)) |>           # convert to within-sample proportion
  ungroup() |>                          # important: drop the sample_id grouping before reshaping
  select(sample_id, phylum, rel) |>
  pivot_wider(names_from = phylum, values_from = rel, values_fill = 0)  # long -> wide; missing = 0

# cor() correlates the columns pairwise -> a phylum x phylum correlation matrix.
cmat <- cor(as.matrix(select(phy_wide, -sample_id)))  # drop the id column; cor() needs a numeric matrix
round(cmat, 2)                                         # peek at the numbers (rounded)

# Visualise that square matrix as a heatmap: reshape it back to long, then tile it.
as.data.frame(cmat) |>
  rownames_to_column("phylum1") |>                         # matrix row names -> a real column
  pivot_longer(-phylum1, names_to = "phylum2", values_to = "r") |>  # one row per phylum-pair
  ggplot(aes(phylum1, phylum2, fill = r)) +
  geom_tile(colour = "white") +
  # gradient2 = a diverging scale: blue (negative) - white (0) - red (positive), centred at 0.
  scale_fill_gradient2(low = "steelblue", mid = "white", high = "firebrick",
                       midpoint = 0, limits = c(-1, 1)) +
  labs(title = "Correlation between phyla (relative abundance)",
       x = NULL, y = NULL, fill = "r") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

The classic gut signal often appears here: **Firmicutes and Bacteroidetes**
correlate negatively — as one rises, the other tends to fall.

---

## 3 · PCA ordination (base R `prcomp`)

Principal Component Analysis summarises many correlated variables into a few
axes. We ordinate samples by their phylum profile and colour by region.

In [ ]:
# prcomp() needs a plain numeric matrix with samples as row names (not a column).
mat <- phy_wide |> column_to_rownames("sample_id") |> as.matrix()

pca <- prcomp(mat, scale. = TRUE)      # PCA; scale. = TRUE standardises each phylum first
var_expl <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)   # % variance captured by each axis

# pca$x holds the sample coordinates; we keep the first two axes (PC1, PC2) for a 2-D plot.
scores <- as.data.frame(pca$x[, 1:2]) |>
  rownames_to_column("sample_id") |>                                  # row names -> a column to join on
  left_join(select(meta, sample_id, nationality), by = "sample_id")   # attach region for colouring

ggplot(scores, aes(PC1, PC2, colour = nationality)) +
  geom_point(alpha = 0.6, size = 1.6) +
  stat_ellipse(level = 0.7) +          # 70% confidence ellipse per region (visual grouping)
  labs(title = "PCA of gut communities (phylum level)",
       x = paste0("PC1 (", var_expl[1], "%)"),   # show variance explained in the axis label
       y = paste0("PC2 (", var_expl[2], "%)"),
       colour = "Region") +
  theme_minimal()

**A second, complementary view — the scree plot.** Before trusting a 2-D
ordination, check *how much* variance PC1 and PC2 actually capture. A **scree
plot** of the per-axis variance answers that in one glance.

In [ ]:
# Turn the var_expl vector into a small data frame, one row per principal component.
scree <- tibble(pc = factor(seq_along(var_expl)), variance = var_expl)

ggplot(head(scree, 8), aes(pc, variance)) +   # first 8 axes are plenty to see the "elbow"
  geom_col(fill = "steelblue") +
  geom_text(aes(label = paste0(variance, "%")), vjust = -0.3, size = 3) +  # label each bar
  labs(title = "Variance explained by each principal component",
       x = "Principal component", y = "% variance") +
  theme_minimal()

> **Instructor note.** PCA (base R) and NMDS (from the capstone, via `vegan`) are
> both ordinations. PCA is fast and linear; NMDS handles ecological
> (Bray–Curtis) distances better. Showing both makes the concept stick.

---

## Recap

- **`geom_tile()`** turns any summarised long table into a **heatmap**.
- `cor()` + a diverging fill scale reveals **relationships between variables**.
- **`prcomp()`** gives a quick PCA ordination — all base R + ggplot2, CRAN-only.

That closes the R refresher. From here, everything in the week is more of the
same verbs and plots on richer data.